In [5]:
import os
import numpy as np
import pandas as pd
from PIL import Image

In [6]:

def extract_and_transform_images(source_folder, target_size=(128, 128)):
    print(f"📂 Scanning '{source_folder}' for unstructured images...\n")
    
    processed_data = []
    
    for filename in os.listdir(source_folder):
        if filename.lower().endswith(('.png', '.jpg', '.jpeg')):
            filepath = os.path.join(source_folder, filename)
            
            try:
                # --- PHASE 1: EXTRACT ---
                with Image.open(filepath) as img:
                    # Capture the original metadata before we change it
                    orig_width, orig_height = img.size
                    orig_mode = img.mode
                    file_size_kb = round(os.path.getsize(filepath) / 1024, 2)
                    
                    # --- PHASE 2: TRANSFORM ---
                    if img.mode != 'RGB':
                        img = img.convert('RGB')
                        
                    img_resized = img.resize(target_size)
                    
                    img_array = np.array(img_resized)
                    
                    flat_array = img_array.flatten().tolist()
                    
                    processed_data.append({
                        "filename": filename,
                        "original_width": orig_width,
                        "original_height": orig_height,
                        "original_format": orig_mode,
                        "file_size_kb": file_size_kb,
                        "image_tensor": flat_array  # The actual mathematical image data!
                    })
                    
                print(f"✅ Transformed: {filename} (Original: {orig_width}x{orig_height} -> New: 128x128)")
                
            except Exception as e:
                print(f"❌ Failed to process {filename}: {e}")
                
    return pd.DataFrame(processed_data)


In [7]:
def load_to_parquet(df, output_filename="structured_image_dataset.parquet"):
    """
    PHASE 3: LOAD
    Saves the structured DataFrame into a highly compressed Parquet file.
    """
    print(f"\n💾 Loading data into Big Data Parquet format...")
    
    if df.empty:
        print("No images were processed. Exiting.")
        return
        
    df.to_parquet(output_filename, engine='pyarrow', index=False)
    
    print(f"🎉 Success! {len(df)} images serialized and saved to '{output_filename}'")
    
    print("\n--- Verifying Parquet File ---")
    verification_df = pd.read_parquet(output_filename)
    print(verification_df[['filename', 'original_width', 'file_size_kb']].head())
    print("-" * 30)
    print(f"The 'image_tensor' column contains flattened arrays of length: {len(verification_df['image_tensor'][0])}")

In [8]:
if __name__ == "__main__":
    folder_name = "raw_images"
    
    if not os.path.exists(folder_name):
        print(f"⚠️ Please create a folder named '{folder_name}' and put some images in it.")
    else:
        structured_df = extract_and_transform_images(source_folder=folder_name)
        load_to_parquet(structured_df)

📂 Scanning 'raw_images' for unstructured images...

✅ Transformed: pexels-ankit-rainloure-1425442-13308431.jpg (Original: 6000x4000 -> New: 128x128)
✅ Transformed: pexels-didsss-5071339.jpg (Original: 4032x3024 -> New: 128x128)
✅ Transformed: photo-1519125323398-675f0ddb6308.jpg (Original: 1080x720 -> New: 128x128)
✅ Transformed: profile.jpeg (Original: 917x917 -> New: 128x128)

💾 Loading data into Big Data Parquet format...
🎉 Success! 4 images serialized and saved to 'structured_image_dataset.parquet'

--- Verifying Parquet File ---
                                      filename  original_width  file_size_kb
0  pexels-ankit-rainloure-1425442-13308431.jpg            6000       1544.93
1                    pexels-didsss-5071339.jpg            4032       1399.59
2         photo-1519125323398-675f0ddb6308.jpg            1080        136.38
3                                 profile.jpeg             917         87.58
------------------------------
The 'image_tensor' column contains flattened